<a href="https://colab.research.google.com/github/fanat503/Induction-Heads-Tinystories/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import time
import math
import torch.nn.functional as F
import numpy as np
device = 'cuda' if torch.cuda.is_available() else 'cpu'
project_dir = '***'
checkpoint_path = '***'


B = 4
T = 1024
grad_accum_steps = 32
max_steps = 20000
warmup_steps = 200
max_lr = 6e-4
min_lr = 6e-5

class DataLoaderLite:
    def __init__(self, B, T, split='train'):
        self.B = B
        self.T = T
        filename = os.path.join(project_dir, f'{split}.bin')

        self.tokens = np.memmap(filename, dtype=np.uint16, mode='r')
        print(f"[{split}] Downloaded {len(self.tokens):,} tokens")
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position + B * T + 1]
        buf = torch.tensor(buf.astype(np.int64), dtype=torch.long)

        x = (buf[:-1]).view(B, T)
        y = (buf[1:]).view(B, T)

        self.current_position += B * T
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_position = 0
        return x, y

def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it + 1) / warmup_steps
    if it > max_steps:
        return min_lr
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

train_loader = DataLoaderLite(B=B, T=T, split='train')
val_loader = DataLoaderLite(B=B, T=T, split='val')

model = GPT(GPTConfig())
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda')

start_step = 0
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_step = checkpoint['step'] + 1
else:
    print("OOps")

for step in range(start_step, max_steps):
    t0 = time.time()

    if step > 0 and (step % 400 == 0 or step == max_steps - 1):
        model.eval()
        val_loss_accum = 0.0
        val_loss_steps = 20
        with torch.no_grad():
            for _ in range(val_loss_steps):
                x_val, y_val = val_loader.next_batch()
                x_val, y_val = x_val.to(device), y_val.to(device)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits, loss = model(x_val, y_val)
                val_loss_accum += loss.detach().item() / val_loss_steps
        print(f" VALIDATION step {step:4d} | val_loss: {val_loss_accum:.4f}")

        checkpoint = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'val_loss': val_loss_accum
        }
        torch.save(checkpoint, checkpoint_path)
        model.train()

    if step > 0 and step % 1000 == 0:
        checkpoint_path_step = os.path.join(
            project_dir, f'gpt2_mega_checkpoint{step}.pt'
        )
        torch.save(checkpoint, checkpoint_path_step)
        print(f"{checkpoint_path_step}")

    optimizer.zero_grad(set_to_none=True)
    loss_accum = 0.0

    for micro_step in range(grad_accum_steps):
        x, y = train_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits, loss = model(x, y)
        loss = loss / grad_accum_steps
        loss_accum += loss.detach()
        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    scaler.step(optimizer)
    scaler.update()

    if device == 'cuda':
      torch.cuda.synchronize()
    t1 = time.time()
    dt = (t1 - t0) * 1000
    tokens_processed = B * T * grad_accum_steps
    tokens_per_sec = tokens_processed / (t1 - t0)

    print(f"step {step:4d} | loss: {loss_accum.item():.4f} | lr: {lr:.4e} | norm: {norm:.2f} | dt: {dt:.2f}ms | tok/sec: {tokens_per_sec:.2f}")